# QuoteGuard Parser Lab

This notebook is the current working surface for QuoteGuard.

Scope for now:

- audit the local PDF corpus
- compare parser outputs
- inspect extracted structure manually
- export parser-lab artefacts to `data/processed/parser_lab`

Out of scope for now:

- retrieval benchmarking
- LLM integration
- guardrails
- app UI


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, UTC
import hashlib
import json
from pathlib import Path
import statistics
import time

from IPython.display import HTML, display
import matplotlib.pyplot as plt
import pandas as pd

try:
    import fitz  # PyMuPDF
except ImportError:
    fitz = None

try:
    import pymupdf4llm
except ImportError:
    pymupdf4llm = None

try:
    from docling.document_converter import DocumentConverter
except ImportError:
    DocumentConverter = None

try:
    from pypdf import PdfReader
except ImportError:
    PdfReader = None


In [ ]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RAW_DIR = ROOT / 'data' / 'raw_pdfs'
PROCESSED_DIR = ROOT / 'data' / 'processed' / 'parser_lab'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PDF_PATHS = sorted(path for path in RAW_DIR.glob('*.pdf') if path.is_file())
ROOT, RAW_DIR, PROCESSED_DIR, len(PDF_PATHS)


## Corpus Audit Helpers

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def page_count(path: Path) -> int | None:
    if fitz is not None:
        return fitz.open(path).page_count
    if PdfReader is not None:
        return len(PdfReader(str(path)).pages)
    return None


def sample_text_metrics(path: Path, sample_pages: int = 5) -> dict:
    if fitz is None:
        return {
            'page_count': page_count(path),
            'sample_characters': None,
            'native_pdf': None,
            'has_multi_column_layout': None,
            'has_repeated_headers_footers': None,
            'has_page_numbers': None,
            'source_version_date': None,
        }

    doc = fitz.open(path)
    pages = min(doc.page_count, sample_pages)
    page_texts = []
    multi_column_flags = []
    headers = []
    footers = []

    for index in range(pages):
        page = doc.load_page(index)
        page_texts.append(page.get_text('text'))
        blocks = [block for block in page.get_text('blocks') if len(block) >= 5 and str(block[4]).strip()]
        if blocks:
            x_positions = sorted({round(block[0], 1) for block in blocks})
            multi_column_flags.append(len(x_positions) >= 2 and max(x_positions) - min(x_positions) > 120)
            ordered = sorted(blocks, key=lambda block: (block[1], block[0]))
            headers.append(str(ordered[0][4]).strip())
            footers.append(str(ordered[-1][4]).strip())

    sample_characters = sum(len(text.strip()) for text in page_texts)
    repeated_header = len(set(value for value in headers if value)) < len([value for value in headers if value]) if headers else False
    repeated_footer = len(set(value for value in footers if value)) < len([value for value in footers if value]) if footers else False
    has_page_numbers = any(str(i + 1) in footer for i, footer in enumerate(footers)) if footers else False
    metadata = doc.metadata or {}

    return {
        'page_count': doc.page_count,
        'sample_characters': sample_characters,
        'native_pdf': sample_characters > 400,
        'has_multi_column_layout': any(multi_column_flags),
        'has_repeated_headers_footers': repeated_header or repeated_footer,
        'has_page_numbers': has_page_numbers,
        'source_version_date': metadata.get('subject') or metadata.get('title') or metadata.get('creationDate') or metadata.get('modDate'),
    }


def layout_profile(metrics: dict) -> dict:
    sample_characters = metrics.get('sample_characters') or 0
    return {
        'text_heavy': sample_characters > 3000,
        'table_heavy': 'manual_review',
        'form_heavy': 'manual_review',
        'image_heavy': sample_characters < 200 if metrics.get('sample_characters') is not None else 'manual_review',
    }


def audit_pdf(path: Path) -> dict:
    metrics = sample_text_metrics(path)
    return {
        'file_name': path.name,
        'file_size_bytes': path.stat().st_size,
        'page_count': metrics['page_count'],
        'source_version_date': metrics['source_version_date'],
        'native_pdf': metrics['native_pdf'],
        'layout_profile': layout_profile(metrics),
        'has_multi_column_layout': metrics['has_multi_column_layout'],
        'has_repeated_headers_footers': metrics['has_repeated_headers_footers'],
        'has_page_numbers': metrics['has_page_numbers'],
        'has_tables_across_page_breaks': 'manual_review',
        'expected_retrieval_difficulty': 'manual_review',
        'checksum_sha256': sha256_file(path),
    }


In [ ]:
corpus_audit_rows = [audit_pdf(path) for path in PDF_PATHS]
corpus_audit_df = pd.DataFrame(corpus_audit_rows)
corpus_audit_df


In [ ]:
corpus_audit_df.to_csv(PROCESSED_DIR / 'corpus_audit.csv', index=False)
(PROCESSED_DIR / 'corpus_audit.json').write_text(json.dumps(corpus_audit_rows, indent=2) + '\n', encoding='utf-8')
PROCESSED_DIR / 'corpus_audit.csv'


## Parser Comparison Helpers

In [ ]:
@dataclass
class ParsedSection:
    heading: str
    text: str
    page_number: int
    section_path: list[str]


def sectionize_markdown(markdown_text: str, page_number: int = 1) -> list[ParsedSection]:
    sections: list[ParsedSection] = []
    current_heading = 'Document'
    current_path = [current_heading]
    buffer: list[str] = []

    def flush() -> None:
        if not buffer:
            return
        text = '\n'.join(buffer).strip()
        if not text:
            return
        sections.append(ParsedSection(current_heading, text, page_number, list(current_path)))
        buffer.clear()

    for raw_line in markdown_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith('#'):
            flush()
            level = len(line) - len(line.lstrip('#'))
            heading = line.lstrip('#').strip() or 'Untitled'
            if level <= 1:
                current_path = [heading]
            else:
                current_path = current_path[: max(0, level - 1)] + [heading]
            current_heading = heading
            continue
        buffer.append(line)
    flush()
    return sections or [ParsedSection('Document', markdown_text.strip() or 'No text extracted', page_number, ['Document'])]


def parse_with_pymupdf4llm(path: Path) -> list[dict]:
    if pymupdf4llm is None:
        raise RuntimeError('pymupdf4llm is not installed')
    pages = pymupdf4llm.to_markdown(str(path), page_chunks=True)
    sections: list[dict] = []
    if isinstance(pages, list):
        for idx, page in enumerate(pages, start=1):
            text = page.get('text', '')
            metadata = page.get('metadata', {})
            page_number = int(metadata.get('page') or metadata.get('page_number') or idx)
            sections.extend(section.__dict__ for section in sectionize_markdown(text, page_number=page_number))
    else:
        sections.extend(section.__dict__ for section in sectionize_markdown(str(pages), page_number=1))
    return sections


def parse_with_docling(path: Path) -> list[dict]:
    if DocumentConverter is None:
        raise RuntimeError('docling is not installed')
    converter = DocumentConverter()
    result = converter.convert(path)
    markdown_text = result.document.export_to_markdown()
    return [section.__dict__ for section in sectionize_markdown(markdown_text, page_number=1)]


def parse_with_pymupdf_text(path: Path) -> list[dict]:
    if fitz is None:
        raise RuntimeError('pymupdf is not installed')
    doc = fitz.open(path)
    sections: list[dict] = []
    for index in range(doc.page_count):
        text = doc.load_page(index).get_text('text').strip()
        if text:
            sections.append({
                'heading': f'Page {index + 1}',
                'text': text,
                'page_number': index + 1,
                'section_path': [f'Page {index + 1}'],
            })
    return sections


PARSER_FUNCTIONS = {
    'pymupdf4llm': parse_with_pymupdf4llm,
    'docling': parse_with_docling,
    'pymupdf_text': parse_with_pymupdf_text,
}


In [ ]:
def run_parser(backend_name: str, pdf_paths: list[Path] | None = None) -> tuple[pd.DataFrame, Path]:
    pdf_paths = pdf_paths or PDF_PATHS
    parser_fn = PARSER_FUNCTIONS[backend_name]
    run_id = datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')
    run_dir = PROCESSED_DIR / f'{run_id}_{backend_name}'
    parsed_dir = run_dir / 'parsed'
    parsed_dir.mkdir(parents=True, exist_ok=True)
    rows = []

    for path in pdf_paths:
        started = time.perf_counter()
        try:
            sections = parser_fn(path)
            duration_ms = (time.perf_counter() - started) * 1000
            payload = {
                'source_pdf': path.name,
                'parser_backend': backend_name,
                'sections': sections,
            }
            output_path = parsed_dir / f'{path.name}.json'
            output_path.write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')
            row = {
                'file_name': path.name,
                'parser': backend_name,
                'parse_time_ms': round(duration_ms, 2),
                'page_count': page_count(path),
                'parse_time_per_page_ms': round(duration_ms / max(1, page_count(path) or 1), 2),
                'sections_detected': len(sections),
                'total_extracted_characters': sum(len(section['text']) for section in sections),
                'tables_detected': 0,
                'failed_pages': 0,
                'empty_pages': 0,
                'warnings_errors': '',
                'output_size_bytes': output_path.stat().st_size,
                'run_dir': str(run_dir),
            }
        except Exception as exc:
            duration_ms = (time.perf_counter() - started) * 1000
            row = {
                'file_name': path.name,
                'parser': backend_name,
                'parse_time_ms': round(duration_ms, 2),
                'page_count': page_count(path),
                'parse_time_per_page_ms': None,
                'sections_detected': 0,
                'total_extracted_characters': 0,
                'tables_detected': 0,
                'failed_pages': 'unknown',
                'empty_pages': 'unknown',
                'warnings_errors': f'{type(exc).__name__}: {exc}',
                'output_size_bytes': 0,
                'run_dir': str(run_dir),
            }
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(run_dir / 'summary.csv', index=False)
    (run_dir / 'summary.json').write_text(json.dumps(rows, indent=2) + '\n', encoding='utf-8')
    return df, run_dir


In [ ]:
backend_frames = {}
backend_run_dirs = {}
for backend in ['pymupdf4llm', 'docling', 'pymupdf_text']:
    frame, run_dir = run_parser(backend)
    backend_frames[backend] = frame
    backend_run_dirs[backend] = run_dir

comparison_df = pd.concat(backend_frames.values(), ignore_index=True)
comparison_df


In [ ]:
comparison_summary = (
    comparison_df.groupby('parser', dropna=False)
    .agg(
        total_parse_time_ms=('parse_time_ms', 'sum'),
        avg_parse_time_ms=('parse_time_ms', 'mean'),
        avg_parse_time_per_page_ms=('parse_time_per_page_ms', 'mean'),
        total_extracted_characters=('total_extracted_characters', 'sum'),
        total_sections=('sections_detected', 'sum'),
    )
    .reset_index()
)
comparison_summary.to_csv(PROCESSED_DIR / 'parser_comparison_summary.csv', index=False)
comparison_summary


## Visual Review

Use these cells to review parser results without digging through raw JSON files.

In [ ]:
visual_df = comparison_df.copy()
summary_view = comparison_summary.copy()
summary_view = summary_view.sort_values('total_parse_time_ms')
display(
    summary_view.style
    .format({
        'total_parse_time_ms': '{:,.2f}',
        'avg_parse_time_ms': '{:,.2f}',
        'avg_parse_time_per_page_ms': '{:,.2f}',
        'total_extracted_characters': '{:,.0f}',
        'total_sections': '{:,.0f}',
    })
    .background_gradient(subset=['total_parse_time_ms', 'avg_parse_time_per_page_ms'], cmap='RdYlGn_r')
    .background_gradient(subset=['total_extracted_characters', 'total_sections'], cmap='YlGn')
)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
plot_df = comparison_summary.sort_values('parser')

axes[0, 0].bar(plot_df['parser'], plot_df['total_parse_time_ms'])
axes[0, 0].set_title('Total Parse Time by Parser')
axes[0, 0].set_ylabel('ms')

axes[0, 1].bar(plot_df['parser'], plot_df['avg_parse_time_per_page_ms'])
axes[0, 1].set_title('Average Parse Time per Page')
axes[0, 1].set_ylabel('ms/page')

axes[1, 0].bar(plot_df['parser'], plot_df['total_extracted_characters'])
axes[1, 0].set_title('Total Extracted Characters')
axes[1, 0].set_ylabel('characters')

axes[1, 1].bar(plot_df['parser'], plot_df['total_sections'])
axes[1, 1].set_title('Total Sections Detected')
axes[1, 1].set_ylabel('sections')

for ax in axes.flat:
    ax.tick_params(axis='x', rotation=15)

fig.tight_layout()
plt.show()


In [ ]:
parse_time_pivot = visual_df.pivot(index='file_name', columns='parser', values='parse_time_ms')
char_pivot = visual_df.pivot(index='file_name', columns='parser', values='total_extracted_characters')
section_pivot = visual_df.pivot(index='file_name', columns='parser', values='sections_detected')

print('Parse time by PDF and parser')
display(parse_time_pivot.style.format('{:,.2f}').background_gradient(cmap='Blues'))

print('Extracted characters by PDF and parser')
display(char_pivot.style.format('{:,.0f}').background_gradient(cmap='YlGn'))

print('Sections detected by PDF and parser')
display(section_pivot.style.format('{:,.0f}').background_gradient(cmap='PuBuGn'))


In [ ]:
problem_rows = visual_df[visual_df['warnings_errors'].fillna('') != '']
problem_rows if not problem_rows.empty else 'No parser failures recorded in the current run set.'


## Manual Inspection Helpers

In [ ]:
def load_sections(run_dir: Path, pdf_name: str) -> list[dict]:
    path = run_dir / 'parsed' / f'{pdf_name}.json'
    payload = json.loads(path.read_text(encoding='utf-8'))
    return payload['sections']


def preview_sections(run_dir: Path, pdf_name: str, limit: int = 8, pages: list[int] | None = None) -> pd.DataFrame:
    sections = load_sections(run_dir, pdf_name)
    if pages is not None:
        page_set = set(pages)
        sections = [section for section in sections if int(section['page_number']) in page_set]
    sections = sections[:limit]
    rows = []
    for section in sections:
        rows.append({
            'heading': section['heading'],
            'page_number': section['page_number'],
            'section_path': ' > '.join(section['section_path']),
            'preview': ' '.join(section['text'].split())[:220],
        })
    return pd.DataFrame(rows)


def side_by_side_parser_preview(
    pdf_name: str,
    parsers: list[str] | None = None,
    pages: list[int] | None = None,
    limit: int = 8,
) -> None:
    parsers = parsers or list(backend_run_dirs)
    blocks = []
    for parser_name in parsers:
        run_dir = backend_run_dirs[parser_name]
        df = preview_sections(run_dir, pdf_name, limit=limit, pages=pages)
        if df.empty:
            html_table = '<p><em>No sections found for this selection.</em></p>'
        else:
            html_table = df.to_html(index=False, escape=False)
        blocks.append(
            f"<div style='flex:1; min-width:320px;'>"
            f"<h4>{parser_name} :: {pdf_name}</h4>{html_table}</div>"
        )
    display(HTML("<div style='display:flex; gap:24px; align-items:flex-start; flex-wrap:wrap;'>" + ''.join(blocks) + '</div>'))


In [ ]:
PDF_PATHS[0].name if PDF_PATHS else None

In [ ]:
sample_pdf = PDF_PATHS[0].name if PDF_PATHS else None
if sample_pdf:
    for backend, run_dir in backend_run_dirs.items():
        print(f'\n=== {backend} :: {sample_pdf} ===')
        display(preview_sections(run_dir, sample_pdf, limit=6))


## Side-by-Side Parser Review

Choose a PDF and page range, then compare parser previews directly next to each other.

In [ ]:
comparison_pdf = next((path.name for path in PDF_PATHS if path.name == 'Allianz_Business_PDS.pdf'), PDF_PATHS[0].name if PDF_PATHS else None)
comparison_pages = [1, 2, 3]
comparison_limit = 6
comparison_parsers = ['pymupdf4llm', 'docling', 'pymupdf_text']
comparison_pdf, comparison_pages, comparison_limit, comparison_parsers


In [ ]:
if comparison_pdf:
    side_by_side_parser_preview(
        comparison_pdf,
        parsers=comparison_parsers,
        pages=comparison_pages,
        limit=comparison_limit,
    )


## Provisional Assessment

These notes are intentionally provisional and based only on the current sample inspection.
Do not lock the parser until you repeat this on the actual target corpus.

In [ ]:
provisional_assessment = pd.DataFrame([
    {
        'parser': 'pymupdf4llm',
        'provisional_position': 'front-runner',
        'what_current_metrics_say': 'Much faster than docling on the current sample, with extracted characters and section counts in the same range.',
        'what_current_preview_says': 'Heading structure looks strong and page assignment appears cleaner than docling on the early pages shown.',
        'what_to_check_next': 'Validate on table-heavy, multi-column, exclusions, and definitions pages before making the final call.',
    },
    {
        'parser': 'docling',
        'provisional_position': 'still viable',
        'what_current_metrics_say': 'Comparable extracted character count and section count, but much slower than pymupdf4llm on this sample.',
        'what_current_preview_says': 'Structured headings look reasonable, but early page assignment may be smeared because several sections appear on page 1.',
        'what_to_check_next': 'See whether docling wins on harder tables or multi-column layouts strongly enough to justify the extra runtime.',
    },
    {
        'parser': 'pymupdf_text',
        'provisional_position': 'baseline only',
        'what_current_metrics_say': 'By far the fastest parser and high raw text extraction, but with far fewer sections detected.',
        'what_current_preview_says': 'Mostly page-level blobs with weak semantic headings, which is poor for later section-aware retrieval.',
        'what_to_check_next': 'Keep only as a speed baseline or fallback, not as the default parser candidate.',
    },
])
display(provisional_assessment.style.set_properties(**{'white-space': 'normal'}))


## Manual Scoring Template

Fill this after inspecting parser outputs.

Suggested criteria:

- reading_order
- heading_preservation
- paragraph_breaks
- bullet_lists
- table_quality
- cross_page_continuity
- header_footer_noise
- page_metadata_preservation


In [ ]:
manual_scoring = pd.DataFrame([
    {
        'pdf_name': path.name,
        'parser': backend,
        'reading_order': None,
        'heading_preservation': None,
        'paragraph_breaks': None,
        'bullet_lists': None,
        'table_quality': None,
        'cross_page_continuity': None,
        'header_footer_noise': None,
        'page_metadata_preservation': None,
        'notes': '',
    }
    for path in PDF_PATHS
    for backend in backend_run_dirs
])
manual_scoring


## Next Move After This Notebook

Do not build retrieval or chat structure until the parser decision is made.

Once the parser is chosen, Phase 2 should be a second pass in this same notebook for chunking experiments.
